# STREEVA Risk Fusion Engine — Motion Anomaly Detection

> **Purpose**: This notebook documents the training of the STREEVA Motion Anomaly model using the public UCI HAR (Human Activity Recognition) dataset. It compares a Random Forest baseline (using 561 pre-extracted features) against a lightweight 1D-CNN (using raw sliding windows) to justify our deployment choice.

---

## 1. Dataset & Scoping
We use the **UCI HAR Dataset**, which contains 3-axial linear acceleration and angular velocity captured at a constant rate of 50Hz, segmented into fixed-width sliding windows of 2.56 seconds (128 readings/window). 

The dataset contains 6 classes:
1. Walking
2. Walking Upstairs
3. Walking Downstairs
4. Sitting
5. Standing
6. Laying

**Honest Scoping (No Fabricated Data)**:
Rather than fabricating synthetic "Fall" data, we group the existing classes into a binary classification task suitable for our risk proxy:
- **Normal (0)**: Walking, Sitting, Standing
- **Anomalous / Irregular Gait (1)**: Walking Upstairs, Walking Downstairs, Laying (sudden orientation change)

*(Note: True fall detection utilizing specialized datasets like SisFall or UMAFall is marked as future work for STREEVA Phase 3).* 


In [ ]:
import os
import urllib.request
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

# Ensure reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# ── 2. Download and Extract UCI HAR Dataset ──────────────────────────────────
dataset_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip"
zip_path = "UCI_HAR_Dataset.zip"
extract_path = "data"

if not os.path.exists(zip_path):
    print("Downloading UCI HAR Dataset...")
    urllib.request.urlretrieve(dataset_url, zip_path)
    print("Download complete.")

if not os.path.exists(os.path.join(extract_path, "UCI HAR Dataset")):
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extraction complete.")
else:
    print("Dataset already extracted.")

base_dir = os.path.join(extract_path, "UCI HAR Dataset")

In [ ]:
# ── 3. Label Mapping ─────────────────────────────────────────────────────────
# Original labels: 1=Walking, 2=Upstairs, 3=Downstairs, 4=Sitting, 5=Standing, 6=Laying
def map_labels(y_raw):
    # Convert to 1D array
    y_raw = y_raw.ravel()
    # 1, 4, 5 -> 0 (Normal)
    # 2, 3, 6 -> 1 (Anomalous)
    y_mapped = np.where(np.isin(y_raw, [1, 4, 5]), 0, 1)
    return y_mapped

# Load Labels
y_train_raw = pd.read_csv(os.path.join(base_dir, 'train', 'y_train.txt'), header=None, sep='\s+').values
y_test_raw = pd.read_csv(os.path.join(base_dir, 'test', 'y_test.txt'), header=None, sep='\s+').values

y_train_binary = map_labels(y_train_raw)
y_test_binary = map_labels(y_test_raw)

print(f"Training samples: {len(y_train_binary)} (Normal: {sum(y_train_binary==0)}, Anomalous: {sum(y_train_binary==1)})")
print(f"Test samples: {len(y_test_binary)} (Normal: {sum(y_test_binary==0)}, Anomalous: {sum(y_test_binary==1)})")

## 4. Baseline Model: Random Forest on 561-Feature Vectors
The UCI HAR dataset provides 561 pre-calculated time and frequency domain variables per window. We'll train a Random Forest on this as our baseline.

In [ ]:
# Load pre-extracted features
X_train_features = pd.read_csv(os.path.join(base_dir, 'train', 'X_train.txt'), header=None, sep='\s+').values
X_test_features = pd.read_csv(os.path.join(base_dir, 'test', 'X_test.txt'), header=None, sep='\s+').values

# Train Random Forest
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_clf.fit(X_train_features, y_train_binary)

rf_preds = rf_clf.predict(X_test_features)

print("Random Forest Baseline Results (561 pre-extracted features):")
print(classification_report(y_test_binary, rf_preds, target_names=['Normal (0)', 'Anomalous (1)']))

# Display Confusion Matrix
cm = confusion_matrix(y_test_binary, rf_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Anomalous'])
disp.plot(cmap='Blues')
plt.title("Random Forest Confusion Matrix")
plt.show()

## 5. Deployment Model: 1D-CNN on Raw Accelerometer Windows

**Why switch from Random Forest to a CNN?**
While the Random Forest achieves excellent accuracy, it relies on 561 complex features extracted over the window (requiring FFTs, kurtosis, skewness, etc.). This makes mobile/edge deployment heavy. Furthermore, Random Forests do not natively compile to **TensorFlow Lite (TFLite)**.

Instead, we will train a lightweight **1D-CNN** directly on the raw `(x, y, z)` accelerometer windows (128 steps x 3 channels). This requires zero manual feature extraction on the device, runs in microseconds, and exports directly to `.tflite`.

In [ ]:
# ── Load Raw Inertial Signals ────────────────────────────────────────────────
def load_signals(subset):
    signals = []
    for axis in ['total_acc_x', 'total_acc_y', 'total_acc_z']:
        filename = f'{axis}_{subset}.txt'
        path = os.path.join(base_dir, subset, 'Inertial Signals', filename)
        df = pd.read_csv(path, header=None, sep='\s+')
        signals.append(df.values)
    # Stack into shape: (samples, timesteps, features)
    return np.dstack(signals)

X_train_raw = load_signals('train')
X_test_raw = load_signals('test')

print(f"Raw Train Shape: {X_train_raw.shape} (samples, 128 timesteps, 3 axes)")
print(f"Raw Test Shape: {X_test_raw.shape}")

In [ ]:
# ── Define and Train 1D-CNN ──────────────────────────────────────────────────
model = Sequential([
    Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=(128, 3)),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid') # Binary classification
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

history = model.fit(
    X_train_raw, y_train_binary, 
    epochs=10, 
    batch_size=32, 
    validation_split=0.2,
    verbose=1
)

In [ ]:
# ── Evaluate CNN ─────────────────────────────────────────────────────────────
cnn_probs = model.predict(X_test_raw)
cnn_preds = (cnn_probs > 0.5).astype(int).flatten()

print("1D-CNN Results (Raw 3-axis windows):")
print(classification_report(y_test_binary, cnn_preds, target_names=['Normal (0)', 'Anomalous (1)']))

cm_cnn = confusion_matrix(y_test_binary, cnn_preds)
disp_cnn = ConfusionMatrixDisplay(confusion_matrix=cm_cnn, display_labels=['Normal', 'Anomalous'])
disp_cnn.plot(cmap='Greens')
plt.title("1D-CNN Confusion Matrix")
plt.show()

In [ ]:
# ── 6. Export to TFLite ──────────────────────────────────────────────────────
models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)
tflite_path = os.path.join(models_dir, "motion_anomaly.tflite")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f"Model exported successfully to: {tflite_path}")
print(f"File size: {os.path.getsize(tflite_path) / 1024:.1f} KB")